In [ ]:
from urllib3.fields import format_multipart_header_param
%load_ext autoreload
%autoreload 2

In [ ]:
import claude_api as api
import json

api.useHaiku()

In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json",
    "solution_criteria": "Key Criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code
* Possible values for "format" are:
** python
** json
** regex

Please generate 3 objects.
"""

    return api.sendMsgGetJson([], prompt)

In [ ]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with json, python or regex
* No comments or explanation
"""
    format = test_case["format"]

    return api.sendMsg([], prompt, startFrom=f"```{format}", stop_sequence="``")

In [ ]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(text, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(text)
    if format == "python":
        return validate_python(text)
    if format == "regex":
        return validate_regex(text)
    return 0


In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    return api.sendMsgGetJson([], eval_prompt)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    syntaxScore = grade_syntax(output, test_case)
    modelGrade = grade_by_model(test_case, output)
    modelScore = modelGrade["score"]
    reasoning = modelGrade["reasoning"]
    score = (syntaxScore + modelScore) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    avgScore = mean([result["score"] for result in results])
    print(f"Avg score: {avgScore}")

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))